In [ ]:
import sympy as sp
# from IPython.display import display

l1, l2 = sp.symbols('l1 l2')
lc1, lc2 = sp.symbols('lc1 lc2')
t1, t2, t3 = sp.symbols('t1 t2 t3')
dt1, dt2, dt3 = sp.symbols('dt1 dt2 dt3')
m1, m2 = sp.symbols('m1 m2')
j1, j2 = sp.symbols('j1 j2') 
g = sp.symbols('g')

In [29]:
st1, ct1 = sp.sin(t1), sp.cos(t1)
st2, ct2 = sp.sin(t2), sp.cos(t2)
st3, ct3 = sp.sin(t3), sp.cos(t3)
s23, c23 = sp.sin(t2+t3), sp.cos(t2+t3)

In [30]:
pc1 = sp.Matrix([
    lc1*ct2*ct1,
    lc1*ct2*st1,
    lc1*st2
])

pc2 = sp.Matrix([
    l1*ct2*ct1 + lc2*c23*ct1,
    l1*ct2*st1 + lc2*c23*st1,
    l1*st2 + lc2*s23
])

pE = sp.Matrix([
    l1*ct2*ct1 + l2*c23*ct1,
    l1*ct2*st1 + l2*c23*st1,
    l1*st2 + l2*s23
])

In [31]:
q = sp.Matrix([t1,t2,t3])
qd = sp.Matrix([dt1,dt2,dt3])

jvL1 = pc1.jacobian(q)
jvL2 = pc2.jacobian(q)
jvE = pE.jacobian(q)

yaw = sp.Matrix([0, 0, 1])
pitch = sp.Matrix([-st1, ct1, 0])
jwL1 = sp.Matrix.hstack(yaw, pitch, sp.zeros(3, 1))
jwL2 = sp.Matrix.hstack(yaw, pitch, pitch)
jwE = sp.Matrix.hstack(yaw, pitch, pitch) 

In [32]:
vc1 = jvL1*qd
vc2 = jvL2*qd
wc1 = jwL1*qd
wc2 = jwL2*qd

In [33]:
K_rot1 = j1*dt2**2 + j1*(dt1*ct2)**2
K_rot2 = j2*(dt2+dt3)**2 + j2*(dt1*c23)**2
K = sp.Rational(1,2)*(m1*vc1.dot(vc1) + m2*vc2.dot(vc2) + K_rot1 + K_rot2)
V = m1*g*pc1[2] + m2*g*pc2[2]

M = sp.simplify(K.diff(qd).jacobian(qd))
G = sp.simplify(V.diff(q))

n = len(q)
C = sp.MutableDenseNDimArray.zeros(n, n, n)
H = sp.Matrix.zeros(n,1)

for i in range(n):
    for j in range(n):
        for k in range(n):
            C[i,j,k] = sp.Rational(1,2) * (
                sp.diff(M[i,j], q[k])
                + sp.diff(M[i,k], q[j])
                - sp.diff(M[j,k], q[i])
            )

for i in range(n):
    expr = 0
    for j in range(n):
        for k in range(n):
            expr += C[i,j,k] * qd[j] * qd[k]

    H[i] = sp.simplify(expr)

display(M)
display(G)
display(H)


Matrix([
[j1*cos(t2)**2 + j2*cos(t2 + t3)**2 + lc1**2*m1*cos(t2)**2 + m2*(l1*cos(t2) + lc2*cos(t2 + t3))**2,                                                            0,                              0],
[                                                                                                0, j1 + j2 + lc1**2*m1 + m2*(l1**2 + 2*l1*lc2*cos(t3) + lc2**2), j2 + lc2*m2*(l1*cos(t3) + lc2)],
[                                                                                                0,                               j2 + lc2*m2*(l1*cos(t3) + lc2),                 j2 + lc2**2*m2]])

Matrix([
[                                                      0],
[g*(lc1*m1*cos(t2) + m2*(l1*cos(t2) + lc2*cos(t2 + t3)))],
[                                  g*lc2*m2*cos(t2 + t3)]])

Matrix([
[-2*dt1*(dt2*(j1*sin(2*t2) + j2*sin(2*t2 + 2*t3) + lc1**2*m1*sin(2*t2) + m2*(l1**2*sin(2*t2) + 2*l1*lc2*sin(2*t2 + t3) + lc2**2*sin(2*t2 + 2*t3)))/2 + dt3*(j2*cos(t2 + t3) + lc2*m2*(l1*cos(t2) + lc2*cos(t2 + t3)))*sin(t2 + t3))],
[                           dt1**2*(j1*sin(2*t2) + j2*sin(2*t2 + 2*t3) + lc1**2*m1*sin(2*t2) + m2*(l1**2*sin(2*t2) + 2*l1*lc2*sin(2*t2 + t3) + lc2**2*sin(2*t2 + 2*t3)))/2 - 2*dt2*dt3*l1*lc2*m2*sin(t3) - dt3**2*l1*lc2*m2*sin(t3)],
[                                                                                                                         dt1**2*(j2*cos(t2 + t3) + lc2*m2*(l1*cos(t2) + lc2*cos(t2 + t3)))*sin(t2 + t3) + dt2**2*l1*lc2*m2*sin(t3)]])